In [2]:
import pandas as pd
from pathlib import Path

In [ ]:
# Load data set
# cleaned data
data = pd.read_csv(r"..\..\..\Datasets\For analysis\full_data_CIFAR100_v3.csv")
# used to generate Results
#data = pd.read_csv(r"data.csv")

# Ensure we only have data with 80/20 datasplit
split_07 = [3911, 3912, 3913, 3914, 3915, 3916, 3917, 3918, 3919, 39110, 39111, 39112, 39113, 39114, 39115]
split_09 = [3901,3902,3903,3904,3905,3906,3907,3908,3909,39010,39011,39012,39013,39014,39015]
split = split_07 + split_09
data = data[~data["exp_id"].isin(split)]

# Rename names for readbility
rename_map = {
    "vit_l_32": "ViT-L/32",
    "vit_b_16": "ViT-B/16",
    "vit_b_32": "ViT-B/32",
    "vit_small_patch16_224": "ViT-S/16",
    "vit_small_patch32_224": "ViT-S/32",
    "vit_tiny_patch16_224": "ViT-T/16",
}

data["model"] = data["model"].replace(rename_map)

In [9]:
# Functions to analyse
def make_subset(df, model, lr, dropout, bs_to_assess):
    # Freeze other hyperparameters so can get all configurations when changing batch rate while freezing other parameters
    subset = df[
        (df["model"] == model) &
        (df["lr"] == lr) &
        (df["batch_size"].isin(bs_to_assess)) &
        (df["dropout"] == dropout) & 
        (df["weight_decay"] == 0.0)
        
    ]
    return subset

def bs_rank(results_df, metric, ascending = False):
    # Rank batch sizes on a given metric, ie "accuracy" or "SPJ"
    df = results_df
    return (
        df.groupby("batch_size")[metric]
        .mean()
        .sort_values(ascending=ascending)
        .reset_index()
        .rename(columns={metric: f"{metric}"})
    )

def summary_table(subset):
    # Summary table that takes mean of every configuration
    view = (
        subset
        .groupby(["batch_size", "model", "lr", "dropout"])
        .agg(
            accuracy=("accuracy", "mean"),
            precision=("precision", "mean"),
            recall=("recall", "mean"),
            specificity=("specificity", "mean"),
            energy=("total_energy_J", "mean"),
            energy_std=("total_energy_J", "std"),
            n_seeds=("accuracy", "nunique"),
            Eg=("Eg","mean"),
            Pg=("Pg", "mean"),
            FPJ=("FPJ", "mean"),
            EDPinv=("EDPinv", "mean"),
            SPJ=("SPJ", "mean"),
        )
        .reset_index()
    )
    print("=========================================")
    print("View for plotting")
    print(view)
    print("=========================================")
    return view



# ViT-b-32


In [10]:
bs_to_assess = [128, 64, 256, 96, 156, 72]
vit_b_32 = make_subset(data,"ViT-B/32",0.03,0.00,bs_to_assess)
vit_b_32 = summary_table(vit_b_32)


vit_b_32 = vit_b_32.dropna()
selected = vit_b_32
print("ViT-B/32")  
Eg = bs_rank(selected,"Eg", ascending=False)
Pg = bs_rank(selected, "Pg", ascending=False)
ac = bs_rank(selected, "accuracy", ascending=False)
re = bs_rank(selected, "recall", ascending=False)
sp = bs_rank(selected, "specificity", ascending=False)
pre = bs_rank(selected, "precision", ascending=False)
fpj = bs_rank(selected, "FPJ", ascending=False)
edpinv = bs_rank(selected, "EDPinv", ascending=False)
spj = bs_rank(selected, "SPJ", ascending=False)
selected["EgPg"] = selected["Pg"] / selected["Eg"]
EgPg = bs_rank(selected,"EgPg", ascending=False)
energy = bs_rank(selected,"energy", ascending=True)

print("==========================================================================================")
print("PERFORMANCE")
print("==========================================================================================")
print("\n")
final = pd.concat([Pg, ac, re, sp, pre], axis=1)
print(final.to_markdown(index=False))


print("==========================================================================================")
print("ENERGY EFFICIENCY")
print("==========================================================================================")
print("\n")
final = pd.concat([Eg, spj, edpinv, fpj, energy], axis=1).dropna()
print(final.to_markdown(index=False))


print("==========================================================================================")
print("ENERGY EFFICIENCY 2")
print("==========================================================================================")
print("\n")
final = pd.concat([EgPg, Eg, Pg, energy], axis=1).dropna()
print(final.to_markdown(index=False))

View for plotting
   batch_size     model    lr  dropout  accuracy  precision    recall  \
0          64  ViT-B/32  0.03      0.0  0.858305   0.858334  0.858423   
1          96  ViT-B/32  0.03      0.0  0.857913   0.857742  0.857766   
2         128  ViT-B/32  0.03      0.0  0.858745   0.858672  0.858760   
3         156  ViT-B/32  0.03      0.0  0.858213   0.858112  0.858082   
4         256  ViT-B/32  0.03      0.0  0.857639   0.857799  0.857583   

   specificity         energy   energy_std  n_seeds        Eg        Pg  \
0     0.998569  153818.934301   906.552554       17  0.006234  0.631524   
1     0.998565  151693.146273   709.096618       14  0.006613  0.630327   
2     0.998573  150172.969067  2683.479873       22  0.006860  0.632350   
3     0.998568  150017.181288  1009.612935       15  0.006850  0.631050   
4     0.998562  149909.139872   812.576130       16  0.006928  0.630016   

             FPJ        EDPinv       SPJ  
0  285991.597541  6.984316e-09  3.120655  
1  289

# Vit-s-32

In [14]:
bs_to_assess = [128, 64, 256, 96, 156, 72]
vit_s_32 = make_subset(data,"ViT-S/32",0.03,0.00,bs_to_assess)
vit_s_32 = summary_table(vit_s_32)
vit_s_32

View for plotting
   batch_size     model    lr  dropout  accuracy  precision    recall  \
0          96  ViT-S/32  0.03      0.0  0.879260   0.879259  0.879276   
1         128  ViT-S/32  0.03      0.0  0.879904   0.879961  0.879803   
2         156  ViT-S/32  0.03      0.0  0.881353   0.881280  0.881370   
3         256  ViT-S/32  0.03      0.0  0.881589   0.881559  0.881524   

   specificity        energy  energy_std  n_seeds        Eg        Pg  \
0     0.998780  79030.703775  532.334410       15  0.016965  0.678955   
1     0.998787  77368.877288  742.197578       16  0.018319  0.680398   
2     0.998802  77812.501887  582.294915       14  0.018138  0.683776   
3     0.998804  85888.733749  917.094673        8  0.013713  0.684282   

             FPJ        EDPinv       SPJ  
0  143974.291271  1.939548e-08  6.073846  
1  147073.456007  2.006533e-08  6.204591  
2  146229.748231  2.010155e-08  6.168997  
3  132486.110190  1.850889e-08  5.589194  


,batch_size,model,lr,dropout,accuracy,precision,recall,specificity,energy,energy_std,n_seeds,Eg,Pg,FPJ,EDPinv,SPJ
0,96,ViT-S/32,0.03,0.0,0.879260,0.879259,0.879276,0.998780,79030.703775,532.334410,15,0.016965,0.678955,143974.291271,1.939548e-08,6.073846
1,128,ViT-S/32,0.03,0.0,0.879904,0.879961,0.879803,0.998787,77368.877288,742.197578,16,0.018319,0.680398,147073.456007,2.006533e-08,6.204591
2,156,ViT-S/32,0.03,0.0,0.881353,0.881280,0.881370,0.998802,77812.501887,582.294915,14,0.018138,0.683776,146229.748231,2.010155e-08,6.168997
3,256,ViT-S/32,0.03,0.0,0.881589,0.881559,0.881524,0.998804,85888.733749,917.094673,8,0.013713,0.684282,132486.110190,1.850889e-08,5.589194


In [16]:
print("ViT-S/32")
bs_to_assess = [128, 64, 256, 96, 156, 72]
vit_s_32 = make_subset(data,"ViT-S/32",0.03,0.00,bs_to_assess)
vit_s_32 = summary_table(vit_s_32)
print("=============================================================================")
print(vit_s_32)
print("=============================================================================")

selected = vit_s_32
Eg = bs_rank(selected,"Eg", ascending=False)
Pg = bs_rank(selected, "Pg", ascending=False)
ac = bs_rank(selected, "accuracy", ascending=False)
re = bs_rank(selected, "recall", ascending=False)
sp = bs_rank(selected, "specificity", ascending=False)
pre = bs_rank(selected, "precision", ascending=False)
fpj = bs_rank(selected, "FPJ", ascending=False)
edpinv = bs_rank(selected, "EDPinv", ascending=False)
spj = bs_rank(selected, "SPJ", ascending=False)
selected["EgPg"] = selected["Pg"] / selected["Eg"]
EgPg = bs_rank(selected,"EgPg", ascending=False)
energy = bs_rank(selected,"energy", ascending=True)


print("==========================================================================================")
print("PERFORMANCE")
print("==========================================================================================")
print("\n")
final = pd.concat([Pg, ac, re, sp, pre], axis=1)
print(final.to_markdown(index=False))


print("==========================================================================================")
print("ENERGY EFFICIENCY")
print("==========================================================================================")
print("\n")
final = pd.concat([Eg, spj, edpinv, fpj, energy], axis=1).dropna()
print(final.to_markdown(index=False))


print("==========================================================================================")
print("ENERGY EFFICIENCY 2")
print("==========================================================================================")
print("\n")
final = pd.concat([EgPg,Eg, Pg, energy], axis=1).dropna()
print(final.to_markdown(index=False))

ViT-S/32
View for plotting
   batch_size     model    lr  dropout  accuracy  precision    recall  \
0          96  ViT-S/32  0.03      0.0  0.879260   0.879259  0.879276   
1         128  ViT-S/32  0.03      0.0  0.879904   0.879961  0.879803   
2         156  ViT-S/32  0.03      0.0  0.881353   0.881280  0.881370   
3         256  ViT-S/32  0.03      0.0  0.881589   0.881559  0.881524   

   specificity        energy  energy_std  n_seeds        Eg        Pg  \
0     0.998780  79030.703775  532.334410       15  0.016965  0.678955   
1     0.998787  77368.877288  742.197578       16  0.018319  0.680398   
2     0.998802  77812.501887  582.294915       14  0.018138  0.683776   
3     0.998804  85888.733749  917.094673        8  0.013713  0.684282   

             FPJ        EDPinv       SPJ  
0  143974.291271  1.939548e-08  6.073846  
1  147073.456007  2.006533e-08  6.204591  
2  146229.748231  2.010155e-08  6.168997  
3  132486.110190  1.850889e-08  5.589194  
   batch_size     model   

# VIT-s-16


In [17]:
bs_to_assess = [128, 64, 256, 96, 156, 72]
vit_s_16 = make_subset(data,"ViT-S/16",0.03,0.00,bs_to_assess)
vit_s_16 = summary_table(vit_s_16)

selected = vit_s_16
print("ViT-S/16")
Eg = bs_rank(selected,"Eg", ascending=False)
Pg = bs_rank(selected, "Pg", ascending=False)
ac = bs_rank(selected, "accuracy", ascending=False)
re = bs_rank(selected, "recall", ascending=False)
sp = bs_rank(selected, "specificity", ascending=False)
pre = bs_rank(selected, "precision", ascending=False)
fpj = bs_rank(selected, "FPJ", ascending=False)
edpinv = bs_rank(selected, "EDPinv", ascending=False)
spj = bs_rank(selected, "SPJ", ascending=False)
selected["EgPg"] = selected["Pg"] / selected["Eg"]
EgPg = bs_rank(selected,"EgPg", ascending=False)

print("==========================================================================================")
print("PERFORMANCE")
print("==========================================================================================")
print("\n")
final = pd.concat([Pg, ac, re, sp, pre], axis=1)
print(final.to_markdown(index=False))

print("==========================================================================================")
print("ENERGY EFFICIENCY")
print("==========================================================================================")
print("\n")
final = pd.concat([Eg, spj, edpinv, fpj, energy], axis=1).dropna()
print(final.to_markdown(index=False))

print("==========================================================================================")
print("ENERGY EFFICIENCY 2")
print("==========================================================================================")
print("\n")
final = pd.concat([EgPg, Eg, Pg, energy], axis=1).dropna()
print(final.to_markdown(index=False))

View for plotting
   batch_size     model    lr  dropout  accuracy  precision    recall  \
0          64  ViT-S/16  0.03      0.0  0.899317   0.899467  0.899353   
1         128  ViT-S/16  0.03      0.0  0.902258   0.902327  0.902277   
2         256  ViT-S/16  0.03      0.0  0.903072   0.903120  0.903109   

   specificity         energy  energy_std  n_seeds        Eg        Pg  \
0     0.998983  163062.551078  419.760953       17  0.005197  0.726765   
1     0.999013  163498.198366  939.474151       17  0.005080  0.733864   
2     0.999021  162183.341288  717.631441       17  0.005260  0.735857   

             FPJ        EDPinv       SPJ  
0  272803.205420  6.471374e-09  2.943674  
1  272083.327022  6.358781e-09  2.935906  
2  274285.460681  6.479306e-09  2.959668  
ViT-S/16
PERFORMANCE


|   batch_size |       Pg |   batch_size |   accuracy |   batch_size |   recall |   batch_size |   specificity |   batch_size |   precision |
|-------------:|---------:|-------------:|-----------:|

# ViT Ti

In [ ]:
bs_to_assess = [128, 64, 256, 96, 156, 72]
vit_t_16 = make_subset(data,"ViT-T/16",0.01,0.00,bs_to_assess)
vit_t_16 = summary_table(vit_t_16)

selected = vit_t_16
print("ViT-Ti/16")
Eg = bs_rank(selected,"Eg", ascending=False)
Pg = bs_rank(selected, "Pg", ascending=False)
ac = bs_rank(selected, "accuracy", ascending=False)
re = bs_rank(selected, "recall", ascending=False)
sp = bs_rank(selected, "specificity", ascending=False)
pre = bs_rank(selected, "precision", ascending=False)
fpj = bs_rank(selected, "FPJ", ascending=False)
edpinv = bs_rank(selected, "EDPinv", ascending=False)
spj = bs_rank(selected, "SPJ", ascending=False)
selected["EgPg"] = selected["Pg"] / selected["Eg"]
EgPg = bs_rank(selected,"EgPg", ascending=False)
energy = bs_rank(selected,"energy", ascending=True)


print("==========================================================================================")
print("PERFORMANCE")
print("==========================================================================================")
print("\n")
final = pd.concat([Pg, ac, re, sp, pre], axis=1)
print(final.to_markdown(index=False))

print("==========================================================================================")
print("ENERGY EFFICIENCY")
print("==========================================================================================")
print("\n")
final = pd.concat([Eg, spj, edpinv, fpj, energy], axis=1).dropna()
print(final.to_markdown(index=False))

print("==========================================================================================")
print("ENERGY EFFICIENCY 2")
print("==========================================================================================")
print("\n")
final = pd.concat([EgPg, Eg, Pg, energy], axis=1).dropna()
print(final.to_markdown(index=False))

View for plotting
   batch_size     model    lr  dropout  accuracy  precision    recall  \
0          64  ViT-T/16  0.01      0.0  0.858039   0.857868  0.858296   
1         128  ViT-T/16  0.01      0.0  0.853467   0.853214  0.853635   
2         256  ViT-T/16  0.01      0.0  0.849850   0.849935  0.849948   

   specificity         energy   energy_std  n_seeds        Eg   Eg_norm  \
0     0.998566   88667.696682   391.849206       18  0.011444  0.031085   
1     0.998520   92500.795673  2964.341669       18  0.010412  0.019927   
2     0.998483  100177.098042   740.339969       18  0.008103  0.001675   

         Pg            FPJ        EDPinv       SPJ  FPJ_norm  EDPinv_norm  \
0  0.630891  132906.258522  1.590432e-08  5.413571  0.050967     0.739826   
1  0.620710  127519.477879  1.568711e-08  5.194155  0.034221     0.728964   
2  0.613033  117640.545039  1.437100e-08  4.791764  0.003509     0.663156   

   SPJ_norm  
0  0.824021  
1  0.781957  
2  0.704815  
ViT-Ti/16
PERFORMANCE

